# 06 — Customer Lifetime Value & Unit Economics

The churn prediction model tells us *who* is likely to leave. This notebook answers the more strategically important follow-on question: **how much is preventing that churn actually worth?**

## The CLV Formula

We use the steady-state CLV formula for subscription businesses:

$$\text{CLV} = \frac{\text{Monthly Revenue} \times \text{Gross Margin}}{\text{Monthly Churn Rate}}$$

This formula assumes constant monthly revenue and constant churn rate — a simplification, but one that produces directionally correct results for customer segmentation and ROI analysis. Breaking it down:
- **Monthly Revenue** = the customer's current `MonthlyCharges`
- **Gross Margin** = assumed at a standard SaaS rate (typically 70–80%)
- **Monthly Churn Rate** = each customer's individual predicted churn probability (from our XGBoost model)

By plugging in each customer's model-predicted churn probability rather than a single aggregate rate, we get a **personalised CLV estimate** — customers the model believes are high-risk have their expected future value appropriately discounted.

## Unit Economics Context

CLV in isolation is only half the picture. We also compute **Customer Acquisition Cost (CAC)** and key unit economics ratios:
- **LTV:CAC ratio** — industry benchmark is ≥ 3:1; below 1:1 means you're losing money on every customer acquired
- **CAC Payback Period** — how many months of gross margin it takes to recoup the acquisition cost; SaaS benchmark is under 12 months

## 1. Setup & Load Data

In [ ]:
import sys
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '../')

from src.preprocessing import get_model_features
from src.clv import (
    calculate_clv,
    add_unit_economics,
    segment_by_clv,
    clv_summary,
    unit_economics_kpis,
    plot_clv_distribution,
    plot_clv_by_contract,
    plot_ltv_cac_scatter,
    plot_payback_distribution
)

# Load processed data
df = pd.read_csv('../data/processed/customers_processed.csv')
print(f'Loaded: {df.shape[0]:,} customers')

# Load saved XGBoost model
with open('../models/xgboost_churn.pkl', 'rb') as f:
    model = pickle.load(f)
print(f'Model loaded: {type(model).__name__}')

## 2. Generate Churn Probability Scores

We run every customer through the trained XGBoost model to get an individualised churn probability. This becomes the denominator in the CLV formula — a customer the model assigns 50% monthly churn probability is expected to generate roughly half the lifetime value of an equivalent customer with 25% churn probability.

In [ ]:
X, y = get_model_features(df)

# Predict churn probabilities for all customers
df['churn_probability'] = model.predict_proba(X)[:, 1]

print('Churn probability distribution:')
print(df['churn_probability'].describe().round(4).to_string())

# Sanity check: high-probability customers should be overrepresented in actual churners
churn_col = 'Churn' if 'Churn' in df.columns else 'churn'
if df[churn_col].dtype == object:
    actual_churn = (df[churn_col] == 'Yes').astype(int)
else:
    actual_churn = df[churn_col].astype(int)

df['actual_churn'] = actual_churn
high_risk_mask = df['churn_probability'] > 0.5
print(f'\nOf customers scored > 50% churn probability: {actual_churn[high_risk_mask].mean():.1%} actually churned.')
print(f'Of customers scored < 50% churn probability: {actual_churn[~high_risk_mask].mean():.1%} actually churned.')

## 3. Calculate CLV and Unit Economics

In [ ]:
# Calculate CLV using the model-predicted churn probability
df = calculate_clv(df)
print('CLV calculation complete.')
print(f'CLV column added. Sample values:')
print(df[['churn_probability', 'clv']].describe().round(2).to_string())

In [ ]:
# Add unit economics: CAC, LTV:CAC ratio, payback period
df = add_unit_economics(df)
print('Unit economics added.')

# Show the new columns
new_econ_cols = [c for c in df.columns if any(kw in c.lower() for kw in ['cac', 'ltv', 'payback', 'clv', 'margin'])]
print(f'\nUnit economics columns: {new_econ_cols}')

In [ ]:
# Segment customers into CLV tiers
df = segment_by_clv(df)
print('CLV segmentation complete.')

if 'clv_segment' in df.columns:
    print('\nCustomers per CLV segment:')
    print(df['clv_segment'].value_counts().sort_index().to_string())

## 4. CLV Summary Table

In [ ]:
summary = clv_summary(df)
print('CLV Summary by Segment:')
display(summary)

## 5. Unit Economics KPIs

In [ ]:
kpis = unit_economics_kpis(df)

print('=== Portfolio Unit Economics KPIs ===')
print()
for key, value in kpis.items():
    if isinstance(value, float):
        if 'ratio' in key.lower() or 'margin' in key.lower():
            print(f'  {key:<35} {value:.2f}x' if 'ratio' in key.lower() else f'  {key:<35} {value:.1%}')
        elif 'payback' in key.lower():
            print(f'  {key:<35} {value:.1f} months')
        elif any(kw in key.lower() for kw in ['clv', 'cac', 'mrr', 'revenue']):
            print(f'  {key:<35} ${value:,.2f}')
        else:
            print(f'  {key:<35} {value:.4f}')
    else:
        print(f'  {key:<35} {value}')

## 6. CLV Distribution

In [ ]:
fig1 = plot_clv_distribution(df)
fig1.show()

## 7. CLV by Contract Type

In [ ]:
fig2 = plot_clv_by_contract(df)
fig2.show()

## 8. LTV:CAC Scatter Plot

This scatter plot plots every customer's LTV against their CAC. Customers above the diagonal (LTV > CAC) are 'profitable to acquire.' The further above the line, the better the unit economics. Any customer below the line represents negative ROI acquisition — worth examining for which channel or segment they came from.

In [ ]:
fig3 = plot_ltv_cac_scatter(df)
fig3.show()

## 9. CAC Payback Period Distribution

In [ ]:
fig4 = plot_payback_distribution(df)
fig4.show()

## 10. Business Impact Scenario

**Scenario: What is the incremental CLV gain if we reduce month-to-month churn by 5 percentage points?**

This is the kind of calculation that turns a data science project into a board-level business case. If we can demonstrate that a specific retention programme — say, a dedicated CSM touchpoint in month 2 and a contract upgrade offer in month 4 — reduces monthly churn by 5pp among month-to-month customers, how much incremental CLV does that generate?

In [ ]:
contract_col = 'Contract' if 'Contract' in df.columns else 'contract'

# Identify month-to-month customers
mtm_mask = df[contract_col].str.lower().str.contains('month', na=False)
df_mtm = df[mtm_mask].copy()
n_mtm = len(df_mtm)

print(f'Month-to-month customers: {n_mtm:,}')
print(f'Current average churn probability: {df_mtm["churn_probability"].mean():.1%}')

charges_col = 'MonthlyCharges' if 'MonthlyCharges' in df_mtm.columns else 'monthly_charges'
gross_margin = 0.75  # standard SaaS assumption
churn_reduction = 0.05  # 5 percentage points

# Current CLV for month-to-month customers
current_churn_proba = df_mtm['churn_probability'].clip(lower=0.001)  # avoid division by zero
current_clv = (df_mtm[charges_col] * gross_margin) / current_churn_proba

# Counterfactual CLV with 5pp lower churn
new_churn_proba = (current_churn_proba - churn_reduction).clip(lower=0.001)
new_clv = (df_mtm[charges_col] * gross_margin) / new_churn_proba

# Incremental CLV
incremental_clv_per_customer = new_clv - current_clv
total_incremental_clv = incremental_clv_per_customer.sum()

print()
print('=== Retention Programme Business Case ===')
print(f'  Month-to-month customers targeted:    {n_mtm:>10,}')
print(f'  Average current churn probability:    {current_churn_proba.mean():>10.1%}')
print(f'  Assumed churn reduction:              {churn_reduction:>10.1%} (5 pp)')
print(f'  Average current CLV:                  ${current_clv.mean():>10,.2f}')
print(f'  Average counterfactual CLV:           ${new_clv.mean():>10,.2f}')
print(f'  Average incremental CLV/customer:     ${incremental_clv_per_customer.mean():>10,.2f}')
print(f'  Total incremental CLV (all MTM):      ${total_incremental_clv:>10,.2f}')
print()
print(f'  If the retention programme costs <${total_incremental_clv:,.0f},')
print(f'  it generates positive ROI.')

In [ ]:
# Sensitivity analysis: how does the business case change at different churn reduction levels?
print('Sensitivity Analysis — Incremental CLV vs. Churn Reduction:')
print(f'{"Churn Reduction":>18} | {"Avg CLV Increase":>18} | {"Total Portfolio Value":>22}')
print('-' * 65)

for reduction in [0.02, 0.05, 0.08, 0.10, 0.15]:
    new_p = (current_churn_proba - reduction).clip(lower=0.001)
    new_c = (df_mtm[charges_col] * gross_margin) / new_p
    incr = (new_c - current_clv)
    avg_incr = incr.mean()
    total_incr = incr.sum()
    print(f'{reduction:>18.0%} | ${avg_incr:>16,.2f} | ${total_incr:>20,.2f}')

## 11. Final Recommendations — Tying It All Together

This final notebook caps a six-part analysis journey: from raw data to churn prediction to CLV quantification. Here's the integrated picture and the three highest-ROI actions the business should take:

---

### The Story in Three Numbers
1. **~26% of customers are on month-to-month contracts and churning at dramatically higher rates** — this single segment represents a disproportionate share of lost MRR each month.
2. **The first 90 days are the highest-risk window** — cohort analysis confirms that customers who survive their first quarter are substantially more likely to stay long-term.
3. **A 5pp reduction in month-to-month churn generates significant incremental CLV** — the business case for retention investment is strong and quantifiable.

---

### Recommendation 1: Deploy the Churn Model in Production
Run the XGBoost model weekly on the active customer base and surface the top 200 highest-risk customers to the Customer Success team each Monday morning. Measure the conversion rate of retention outreach over 90 days to validate the model's real-world lift against a control group.

### Recommendation 2: Build a Contract Upgrade Campaign for High-CLV Month-to-Month Customers
Not all month-to-month customers are equal — some have high monthly charges and low predicted churn (high CLV), while others are high-risk and low-spend (low CLV). Prioritise the upgrade campaign for customers in the top CLV quartile who are also month-to-month: converting these to annual contracts protects the most valuable revenue.

### Recommendation 3: Invest in a Structured 90-Day Onboarding Programme
The cohort analysis and SHAP tenure findings both point to the same intervention: reduce early-period churn. A structured onboarding programme with proactive check-ins at days 14, 30, 60, and 90 costs relatively little but addresses the highest-risk period in a customer's lifecycle. Measure success via month-3 retention rate improvement quarter-over-quarter.

---

### Monitoring & Next Steps
- **Retrain the churn model quarterly** as customer behaviour and product features evolve — model staleness is a real risk in fast-moving SaaS environments.
- **Build a real-time CLV dashboard** that surfaces LTV:CAC by acquisition channel, enabling the marketing team to shift budget toward the highest-quality customer sources.
- **Expand the CLV model to include expansion revenue** — the current formula uses only base monthly charges. Adding upsell probability estimates would give a more complete picture of total customer value potential.